# Week 5 & 6 Deliverables   

## 1. Download Data

### Samples
- [Short-read Illumina (**interleaved** paired-end FASTQ)](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2)
- [Long-read PacBio](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2)

### Reference Genome
The hg38 (or GRCh38) version of the human genome, focusing on the chromosome that contains these genes ([chromosome 10](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz)): 
- [CYP2C8](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A95036772%2D95069497&hgsid=3290464905_o8n5jQXlACoTC3asJu2IgMaUIkFs) (regulates many drugs, including anticancer, diabetes and blood pressure drugs)
- [CYP2C9](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94938658%2D94990091&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates many common drugs, including warfarin / Coumadin and NSAIDs such as Advil)
- [CYP2C19](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94762681%2D94855547&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates… yup, many common drugs, including antiplatelet drugs, antidepressants and anti-epileptic drugs).            

|Genes | CYP2C8 | CYP2C9 | CYP2C19 |
| --- | --- | --- | ---|
| Genomic sequence | chr10:95036772-95069497 | chr10:94938658-94990091 | chr10:94762681-94855547 |
| Strand | - | + | + | 
| Genomic size | 32726 | 51434 | 92867 |

In [ ]:
!mkdir -p data

# SAMPLES
# Download Illumina and PacBio data
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2
!bunzip2 data/*.bz2

In [ ]:
# REFERENCE GENOME 
# chr10 containing CYP2C genes
# Downloading important genes 

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests

# Output FASTA file
output_file = "data/reference_genome.fa"

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

# UCSC FASTA API template
ucsc_fasta_url = "https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chr};start={start};end={end}"

records = []

for gene, info in GENE_INFO.items():
    url = ucsc_fasta_url.format(chr=info["chr"], start=info["start"]-1, end=info["end"])
    r = requests.get(url)
    r.raise_for_status()
    seq = r.json()["dna"]

    # Create SeqRecord
    record = SeqRecord(
        Seq(seq),
        id=gene,
        description=""
    )
    records.append(record)

# Write all genes to a single FASTA
with open(output_file, "w") as f:
    SeqIO.write(records, f, "fasta")

## 2. Align Samples to Reference Genome

**Short-read Illumina (interleaved paired-end FASTQ)**  
- -x: applies multiple options at the same time 
- sr: short read alignment without slicing

**Long-read PacBio**  
- map-hifi: align PacBio high-fidelity reads to a reference genome


In [ ]:
# minimap index
!minimap2 -d data/reference_genome.mmi data/reference_genome.fa 

# Short read Illumina 
!minimap2 -ax sr data/reference_genome.mmi data/illumina.fq > data/illumina.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/reference_genome.mmi data/pacbio.fq > data/pacbio.sam

In [62]:
# Convert SAM to sorted BAM
!samtools view -bS data/illumina.sam | samtools sort -o data/illumina.bam
!samtools view -bS data/pacbio.sam | samtools sort -o data/pacbio.bam

# Index BAM for random access
!samtools index -b data/illumina.bam
!samtools index -b data/pacbio.bam

## 3. Variant Calling 
`bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz`
- mpileup part generates genotype likelihoods at each genomic position with coverage
- call part makes the actual calls to interpret above likelihoods and detect variants 
- -m switch tells the program to use the default calling method (multiallelic caller)
- -v option asks to output only variant sites
- --ploidy 2 options assume that the genome is diploid 
- -O option selects the output format -z selects the vcf.gz format 
- -o output file name  
- Do not waste computer’s time by making mpileup convert from the internal binary representation (BCF) to text (VCF), only to be immediately converted back to binary representation by call. Instead, use -Ou to work with uncompressed BCF output

`bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz`
- -m -any options splits multialleleic into one record per ALT 
- -f option uses the reference genome for allele normalization 

In [63]:
# Index reference genome
!samtools faidx data/reference_genome.fa

In [ ]:
# Call variants 
!bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz
!bcftools index data/illumina.norm.splitted.vcf.gz

!bcftools mpileup -Ou -f data/reference_genome.fa data/pacbio.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/pacbio.norm.splitted.vcf.gz data/pacbio.vcf.gz
!bcftools index data/pacbio.norm.splitted.vcf.gz

!bcftools convert -O v data/illumina.norm.splitted.vcf.gz > data/illumina.norm.splitted.vcf
!bcftools convert -O v data/pacbio.norm.splitted.vcf.gz > data/pacbio.norm.splitted.vcf

## 4. Phase Variant VCFs 

In [ ]:
!extractHAIRS --bam data/illumina.bam --VCF data/illumina.norm.splitted.vcf --out data/illumina.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina.fragments --VCF data/illumina.norm.splitted.vcf --output data/illumina.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.bam --VCF data/pacbio.norm.splitted.vcf --out data/pacbio.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio.fragments --VCF data/pacbio.norm.splitted.vcf --output data/pacbio.hapcut

## 5. Variant Analysis

- After comparing the alleles between the Illumina's phased VCF and the PacBio's phased VCF, there are 783 variants shared between the VCFs, 1368 variants are unique to Illumina, and 1229 variants are unique to PacBio.

Sampled variants that are not common between two technologies:

#### 1. CYP2C19:54675 T->C  GT=T|C DP=258 PQ=100 (Illumina only)

**VCF**

Illumina:
`CYP2C19	54675	.	T	C	90.4009	.	DP=258;VDB=2.32904e-10;SGB=-0.693147;RPBZ=-2.91018;MQBZ=0.907493;MQSBZ=-0.413952;BQBZ=0.271577;SCBZ=-2.7857;MQ0F=0.108527;AC=1;AN=2;DP4=35,99,27,94;MQ=9	GT:PL:AD	0/1:123,0,214:134,121`

PacBio:
`No results`

**BAM**

`Site CYP2C19:54675 ref=t
data/illumina.bam: counts={'C': 169, 'T': 191}, mean_MQ=10.1, softclips=88
data/pacbio.bam: counts={'T': 14, 'C': 1}, mean_MQ=50.266666666666666, softclips=15`

- Illumina: 
    - has roughly 50-50 mix of T/C but the mapping quality (MQ) is very low (`MQ=9`)
    - high softclip counts (88) suggests that this location has some structural variants  that lead to misalignment 
- PacBio:
    - almost all reference nucleotide, except for 1, are T
    - relatively high mean MQ (`mean_MQ=50.2666`) and fewer soft clip reads (15) 

--> Sequencing-related artifacts caused by low mapping quality and misalignment with Illumina technology. 

#### 2. CYP2C9:18070 A->G  GT=A|G DP=277 PQ=100 (Illumina only)

**VCF**
Illumina:
`CYP2C9	18070	.	A	G	222.196	.	DP=277;VDB=7.13142e-09;SGB=-0.693147;RPBZ=-0.966805;MQBZ=-4.26651;MQSBZ=-1.01176;BQBZ=-0.818484;SCBZ=2.27572;MQ0F=0;AC=1;AN=2;DP4=82,67,80,45;MQ=34	GT:PL:AD	0/1:255,0,255:149,124`

PacBio:
`No results`

**BAM**

`Site CYP2C9:18070 ref=a
data/illumina.bam: counts={'A': 307, 'G': 264}, mean_MQ=31.220665499124344, softclips=84
data/pacbio.bam: counts={'A': 15, '<del>': 1, 'G': 2}, mean_MQ=51.588235294117645, softclips=17`

- Illumina: 
    - VDB (Variant Distance Bias for filtering splice-site artefacts in RNA-seq data) is very low (`VDB=7013142e-09`) suggests that there is highly likely a bias and/or a misalignment at this position 
- PacBio:
    - Very low count of alternate G reads (only 2)

--> Sequencing-related artifacts caused by variant distance bias with Illumina technology.

#### 3. CYP2C8:9977 TAAAA->T  GT=TAAAA/T DP=162 PQ=None (PacBio only)

**VCF**

Illumina:
`CYP2C8	9977	.	Taaaaaaaaaaaaaaaaaaaa	Taaaaaaaaaaaaaaaaaa	210.72	.	INDEL;IDV=16;IMF=0.484848;DP=33;VDB=0.954584;SGB=-0.688148;RPBZ=-1.53324;MQBZ=0;MQSBZ=0;BQBZ=-1.66256;SCBZ=0;MQ0F=0;AC=1;AN=2;DP4=13,3,6,9;MQ=60	GT:PL:AD	0/1:244,0,54:16,14`

PacBio:
`CYP2C8	9977	.	Taaaaaaaaaaaaaaaaaaaa	Taaaaaaaaaaaaaaaaa,Taaaaaaaaaaaaaaaa	194.521	.	INDEL;IDV=136;IMF=0.839506;DP=162;VDB=0.852802;SGB=-0.693147;RPBZ=0.707378;MQBZ=0;MQSBZ=0;BQBZ=0.213133;SCBZ=-0.955956;MQ0F=0;AC=1,1;AN=2;DP4=16,10,36,49;MQ=60	GT:PL:AD	1/2:255,90,255,72,0,247:26,26,19`

**BAM**

`Site CYP2C8:9977 ref=T
data/illumina.bam: counts={'T': 29}, mean_MQ=60.0, softclips=3
data/pacbio.bam: counts={'T': 162}, mean_MQ=60.0, softclips=162`

The site lies inside a repeated region of 'a'. The `INDEL` in VCF data for both technology also suggests that there is an indel variant (insertion/deletion of one more nucleotides) at this site, which could lead to lost of genomic information, and potentially lead to disease. 

- PacBio:
    - high softclip counts (162) suggests that this location has some structural variants  that lead to misalignment

--> Wequencing-related artifacts caused by indel variant.


In [71]:
import pysam 

# Phased VCFs
illumina_VCF = "data/illumina.hapcut.phased.VCF"
pacbio_VCF = "data/pacbio.hapcut.phased.VCF"

# Sorted BAM
illumina_BAM = "data/illumina.bam" 
pacbio_BAM = "data/pacbio.bam"

# Reference genome
ref = "data/reference_genome.fa"

def read_variants(vcf_path):
    vcf = pysam.VariantFile(vcf_path)
    variants = {}

    for rec in vcf.fetch():
        for alt in rec.alts:
            key = (rec.chrom, rec.pos, rec.ref, alt)

            # Genotype
            samples = list(rec.samples)
            gt = None
            pq = None
            ps = None
            if samples:
                s = rec.samples[samples[0]]
                # Genotype
                gt_idx = s.get("GT")
                if gt_idx:
                    alleles = [rec.alleles[i] if i is not None else "." for i in gt_idx]
                    gt = "|".join(alleles) if s.phased else "/".join(alleles)

                # Phasing quality (PQ)
                pq = s.get("PQ")

                # Phase set (PS)
                ps = s.get("PS")

            # Depth (DP)
            dp = None
            if samples and "DP" in rec.samples[samples[0]]:
                dp = rec.samples[samples[0]]["DP"]
            elif "DP" in rec.info:
                dp = rec.info["DP"]

            variants[key] = {
                "CHROM": rec.chrom,
                "POS": rec.pos,
                "REF": rec.ref,
                "ALT": alt,
                "GT": gt,
                "DP": dp,
                "PQ": pq,
                "PS": ps,
            }
    return variants
        
illumina_variants = read_variants(illumina_VCF)
pacbio_variants = read_variants(pacbio_VCF)

illumina_keys = set(illumina_variants.keys())
pacbio_keys = set(pacbio_variants.keys())

# Compare
shared_variants = illumina_keys & pacbio_keys
illumina_only = illumina_keys - pacbio_keys
pacbio_only = pacbio_keys - illumina_keys

print(f"Shared variants: {len(shared_variants)}")
print(f"Unique to Illumina: {len(illumina_only)}")
print(f"Unique to PacBio:  {len(pacbio_only)}\n")

Shared variants: 783
Unique to Illumina: 1368
Unique to PacBio:  1229



In [75]:
# Choose variants to inspect
def pick_vars(keys, vars, n=3):
    keys = list(keys)
    def score(k):
        info = vars[k]
        dp = info.get("DP") or 0
        pq = info.get("PQ") or 0
        ps = info.get("PS") or "."
        return (dp, pq, ps)
    keys_sorted = sorted(keys, key=score, reverse=True)
    return keys_sorted[:n]

pick_illumina = pick_vars(illumina_only, illumina_variants, n=3)
pick_pacbio = pick_vars(pacbio_only, pacbio_variants, n=3)

print("\nExample discordant variants (Illumina-only):")
for k in pick_illumina:
    info = illumina_variants[k]
    print(f"{info['CHROM']}:{info['POS']} {info['REF']}->{info['ALT']}  GT={info['GT']} DP={info['DP']} PQ={info['PQ']} PS={info['PS']}")

print("\nExample discordant variants (PacBio-only):")
for k in pick_pacbio:
    info = pacbio_variants[k]
    print(f"{info['CHROM']}:{info['POS']} {info['REF']}->{info['ALT']}  GT={info['GT']} DP={info['DP']} PQ={info['PQ']} PS={info['PS']}")


Example discordant variants (Illumina-only):
CYP2C9:18070 A->G  GT=A|G DP=277 PQ=100 PS=16311
CYP2C19:54675 T->C  GT=T|C DP=258 PQ=100 PS=53980
CYP2C9:16508 C->T  GT=C|T DP=257 PQ=100 PS=16311

Example discordant variants (PacBio-only):
CYP2C8:9977 TAAAA->T  GT=TAAAA/T DP=162 PQ=None PS=None
CYP2C8:9977 TAAA->T  GT=T/TAAA DP=162 PQ=None PS=None
CYP2C8:32291 CAA->C  GT=CAA/C DP=133 PQ=None PS=None


In [ ]:
# generate_igv_batch.py
import os

# Output batch file
batch_file = "igv_commands.txt"

# Snapshot directory
snapshot_dir = "snapshots"
os.makedirs(snapshot_dir, exist_ok=True)

# BAM files
bams = ["data/illumina.bam", "data/pacbio.bam"]

# Reference genome
reference_fasta = "data/reference_genome.fa"

# Discordant variants: (gene, position)
discordant_variants = [("CYP2C19", 54675), ("CYP2C9", 18070), ("CYP2C8", 9977)]

with open(batch_file, "w") as f:
    f.write("new\n")
    f.write(f"genome {reference_fasta}\n")
    
    for bam in bams:
        f.write(f"load {bam}\n")
    
    f.write(f"snapshotDirectory {snapshot_dir}\n\n")
    
    # Go to each variant and take a snapshot
    for gene, pos in discordant_variants:
        start = pos - 10
        end = pos + 10
        f.write(f"goto {gene}:{start}-{end}\n")
        f.write("sort base\n")  
        f.write("collapse\n")   
        f.write(f"snapshot {gene}_{pos}.png\n\n")
    
    f.write("exit\n")

In [ ]:
!igv -b igv_commands.txt

In [60]:
from IPython.display import HTML

html = """
<h3>IGV Snapshots for Discordant Variants</h3>
<div style='display:flex; gap:20px; flex-wrap:wrap;'>
  <figure>
    <figcaption><b>CYP2C19 (54675)</b> — Illumina-only variant</figcaption>
    <img src='snapshots/CYP2C19_54675.png' width='800'>
  </figure>
  <figure>
    <figcaption><b>CYP2C9 (18070)</b> — Illumina-only variant</figcaption>
    <img src='snapshots/CYP2C9_18070.png' width='800'> 
  </figure>
  <figure>
    <figcaption><b>CYP2C8 (9977)</b> — PacBio-only variant</figcaption>
    <img src='snapshots/CYP2C8_9977.png' width='800'>
  </figure>
</div>
"""
display(HTML(html))

In [57]:
import pysam
from collections import Counter
from Bio import SeqIO

fasta_path = "data/reference_genome.fa"
illumina_bam = "data/illumina.bam"
pacbio_bam  = "data/pacbio.bam"

ref_seqs = {r.id: str(r.seq) for r in SeqIO.parse(fasta_path, "fasta")}

queries = [
    ("CYP2C19", 54675),
    ("CYP2C9", 18070),
    ("CYP2C8", 9977),
]

def allele_summary(bam_path, chrom, pos, ref_base=None):
    sam = pysam.AlignmentFile(bam_path, "rb")
    start0 = pos - 1
    end0 = pos
    counts = Counter()
    fwd = Counter()
    rev = Counter()
    mq_totals = []
    softclips = 0
    for pileupcolumn in sam.pileup(chrom, start0, end0, truncate=True, stepper="all"):
        if pileupcolumn.pos != start0:
            continue
        for pr in pileupcolumn.pileups:
            if pr.is_del or pr.is_refskip:
                counts["<del>"] += 1
                continue
            base = pr.alignment.query_sequence[pr.query_position]
            counts[base] += 1
            if pr.alignment.is_reverse:
                rev[base] += 1
            else:
                fwd[base] += 1
            mq_totals.append(pr.alignment.mapping_quality)
            # detect softclip presence anywhere in read
            if any(c[0] == 4 for c in (pr.alignment.cigartuples or [])):
                softclips += 1
    sam.close()
    return {
        "counts": dict(counts),
        "strand_fwd": dict(fwd),
        "strand_rev": dict(rev),
        "mean_MQ": sum(mq_totals)/len(mq_totals) if mq_totals else None,
        "softclip_reads": softclips
    }

for gene, pos in queries:
    ref_base = None
    # obtain reference base from ref_seqs if possible
    seq = ref_seqs.get(gene)
    if seq:
        ref_base = seq[pos-1]
    print(f"\nSite {gene}:{pos} ref={ref_base}")
    for bam in (illumina_bam, pacbio_bam):
        try:
            out = allele_summary(bam, gene, pos, ref_base)
            print(f"{bam}: counts={out['counts']}, mean_MQ={out['mean_MQ']}, softclips={out['softclip_reads']}")
        except Exception as e:
            print("error", e)



Site CYP2C19:54675 ref=t
data/illumina.bam: counts={'C': 169, 'T': 191}, mean_MQ=10.1, softclips=88
data/pacbio.bam: counts={'T': 14, 'C': 1}, mean_MQ=50.266666666666666, softclips=15

Site CYP2C9:18070 ref=a
data/illumina.bam: counts={'A': 307, 'G': 264}, mean_MQ=31.220665499124344, softclips=84
data/pacbio.bam: counts={'A': 15, '<del>': 1, 'G': 2}, mean_MQ=51.588235294117645, softclips=17

Site CYP2C8:9977 ref=T
data/illumina.bam: counts={'T': 29}, mean_MQ=60.0, softclips=3
data/pacbio.bam: counts={'T': 162}, mean_MQ=60.0, softclips=162


## 6. Star-Allele Calls
- Can you figure out the star-allele for each gene of interest? The star-allele database can be found in PharmVar; see this for CYP2C19. Your answer should be something like CYP2C19*12 because X, Y and Z. This step does not have to be automated, but should be at least explained in the notebook.
    - Hint: use phased data!

- Expected output: Jupyter cell(s) with discussion (and code, if you want to do it that way).

In [87]:
from collections import defaultdict

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

def rel_to_abs(chrom, rel_pos):
    gene = GENE_INFO[chrom]
    if gene["strand"] == "+":
        return gene["start"] + rel_pos - 1
    else:
        return gene["end"] - rel_pos + 1
    
def add_absolute_positions(variants):
    abs_variants = {}
    for key, v in variants.items():
        v_copy = v.copy()
        v_copy["ABS_POS"] = rel_to_abs(v_copy["CHROM"], v_copy["POS"])
        abs_variants[key] = v_copy
    return abs_variants

def group_by_ps(variants):
    ps_groups = {}
    for v in variants.values():
        ps = v["PS"] 
        if ps is None:  
            continue
        if ps not in ps_groups:
            ps_groups[ps] = []
        ps_groups[ps].append(v)
    return ps_groups


illumina_variants_abs = add_absolute_positions(illumina_variants)
pacbio_variants_abs  = add_absolute_positions(pacbio_variants)

illumina_ps_groups = group_by_ps(illumina_variants_abs)
pacbio_ps_groups = group_by_ps(pacbio_variants_abs)

print("\n=== Illumina Phase Sets ===")
for ps, group in illumina_ps_groups.items():
    print(f"=== Phase Set {ps} ===")
    for v in sorted(group, key=lambda x: x["POS"]):
        print(f"{v['CHROM']}: {v['ABS_POS']}{v['REF']}>{v['ALT']} GT={v['GT']} DP={v['DP']} PQ={v['PQ']}")

print("\n=== PacBio Phase Sets ===")
for ps, group in pacbio_ps_groups.items():
    print(f"=== Phase Set {ps} ===")
    for v in sorted(group, key=lambda x: x["POS"]):
        print(f"{v['CHROM']}: {v['ABS_POS']}{v['REF']}>{v['ALT']} GT={v['GT']} DP={v['DP']} PQ={v['PQ']}")


=== Illumina Phase Sets ===
=== Phase Set 10170 ===
CYP2C19: 94772850T>C GT=T|C DP=32 PQ=100
CYP2C19: 94772907G>A GT=G|A DP=92 PQ=100
CYP2C19: 94772931G>A GT=A|G DP=70 PQ=100
CYP2C19: 94772937G>A GT=G|A DP=76 PQ=100
CYP2C19: 94773008G>C GT=G|C DP=71 PQ=100
=== Phase Set 11081 ===
CYP2C19: 94773761T>C GT=T|C DP=58 PQ=100
CYP2C19: 94773764T>C GT=T|C DP=57 PQ=100
CYP2C19: 94773766A>C GT=A|C DP=58 PQ=100
CYP2C19: 94773778G>A GT=G|A DP=58 PQ=100
CYP2C19: 94773934T>A GT=T|A DP=73 PQ=100
CYP2C19: 94773943T>G GT=T|G DP=71 PQ=100
CYP2C19: 94773970T>C GT=T|C DP=61 PQ=100
CYP2C19: 94773971G>A GT=G|A DP=61 PQ=100
CYP2C19: 94773981T>C GT=T|C DP=55 PQ=100
=== Phase Set 12467 ===
CYP2C19: 94775147C>T GT=T|C DP=58 PQ=100
CYP2C19: 94775150G>A GT=A|G DP=61 PQ=100
CYP2C19: 94775158T>A GT=A|T DP=59 PQ=100
CYP2C19: 94775367A>C GT=C|A DP=57 PQ=100
CYP2C19: 94775378T>C GT=C|T DP=50 PQ=100
CYP2C19: 94775395G>C GT=C|G DP=54 PQ=100
CYP2C19: 94775438C>G GT=G|C DP=48 PQ=100
CYP2C19: 94775448G>T GT=T|G DP=52 PQ=1